<a href="https://colab.research.google.com/github/arildbn/bban4040/blob/main/martra-notebooks/3-4_medical-assistant-agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div>
    <h1>Large Language Models Projects</a></h1>
    <h3>Apply and Implement Strategies for Large Language Models</h3>
    <p>by <b>Pere Martra</b></p>
    <h2>3.4-Create a Medical Assistant RAG System chat with LangChain & ChromaDB</h2>
</div>



#Installing libraries & Loading Dataset

In [ ]:
# === Colab dependency guard (bban4040) ===
# The langchain / langsmith stack upgrades transitive packages (requests,
# opentelemetry-*) past the exact versions Colab's preinstalled google
# packages pin (google-colab, google-adk, the otlp/gcp exporters), which
# prints noisy "pip's dependency resolver ... is incompatible" errors.
# We pin those families to the versions already installed so the installs
# below leave them untouched. PIP_CONSTRAINT is honored by every %pip call
# in this kernel. Harmless off Colab (nothing matches / nothing to pin).
import os, tempfile
from importlib import metadata

_keep = []
for _dist in metadata.distributions():
    _name = (_dist.metadata.get("Name") or "").strip()
    if not _name:
        continue
    if _name.lower() == "requests" or _name.lower().startswith("opentelemetry"):
        _keep.append(f"{_name}=={_dist.version}")

if _keep:
    _con = os.path.join(tempfile.gettempdir(), "bban4040_pip_constraints.txt")
    with open(_con, "w") as _f:
        _f.write("\n".join(sorted(set(_keep))) + "\n")
    os.environ["PIP_CONSTRAINT"] = _con
    print(f"Pinned {len(_keep)} package(s) to avoid Colab pip resolver conflicts.")


In [ ]:
# This notebook runs on the langchain 1.x line (langchain + langchain-classic).
# langchain-community is NOT installed: it requires langchain-core<1.0 and would
# conflict with the 1.x core. Chroma now lives in its own langchain-chroma
# package, and the DataFrameLoader is replaced with direct Document construction.
%pip install -q langchain
%pip install -q langchain-classic
%pip install -q langchain-openai
%pip install -q langchain-chroma
%pip install -q datasets

In [ ]:
# === portable-setup (bban4040) ===
# Secrets resolve from Colab "Secrets" (userdata) on Colab, or environment
# variables / a local .env file when running locally. Nothing is hardcoded.
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass


def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass
    value = os.environ.get(name, default)
    return value.strip() if isinstance(value, str) else value


_key = get_secret("OPENAI_API_KEY")
if _key:
    os.environ["OPENAI_API_KEY"] = _key


We will download the dataset from the Hugging Face datasets library. It's a dataset with information about diseases.

In [ ]:
from datasets import load_dataset

data = load_dataset("keivalya/MedQuad-MedicalQnADataset", split='train')


In [ ]:
data = data.to_pandas()
data.head(10)

In [ ]:
#uncoment this line if you want to limit the size of the data.
data = data[0:100]

As you can see, the medical information in the dataset is well-organized, and to someone like me, who is not an expert in the field, it appears to be quite valuable. This information could be a useful addition to any general medicine book to support primary care doctors.

Load the langchain libraries to load the document.

In [ ]:
# Chroma moved out of langchain-community into the dedicated langchain-chroma
# package; Document comes from langchain-core (replaces the community DataFrameLoader).
from langchain_chroma import Chroma
from langchain_core.documents import Document

The Document is in the Answer column, and the others columns are Metadata.

In [ ]:
# Equivalent of DataFrameLoader(data, page_content_column="Answer"): one Document
# per row, page content from the "Answer" column and every other column as metadata.
df_document = [
    Document(
        page_content=row["Answer"],
        metadata={col: row[col] for col in data.columns if col != "Answer"},
    )
    for _, row in data.iterrows()
]

In [ ]:
display(df_document[:2])

We can chunk the documents. The size to which we want to split the document is a design decision. The larger it is, the larger the prompt will be, and the slower the Model's response process.

We also need to consider the maximum prompt size and ensure that the document does not exceed it.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size=1250,
                                      separator="\n",
                                      chunk_overlap=100)
texts = text_splitter.split_documents(df_document)


These warnings we see are because it can't perform the partition of the required size. This is because it waits for a page break to divide the text and does so when possible.

In [ ]:
first_doc = texts[1]
print(first_doc.page_content)

### Initialize the Embedding Model and Vector DB

We load the text-embedding-ada-002 model from OpenAI.

In [ ]:
from getpass import getpass
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API Key: ")

In [ ]:
from langchain_openai import OpenAIEmbeddings

model_name = 'text-embedding-ada-002'

embed = OpenAIEmbeddings(
    model=model_name,
    openai_api_key=OPENAI_API_KEY
)

The execution of this cell may take 3 to 5 minutes. If you want it to be faster, you can reduce the number of records in the dataset.

In [ ]:
# Portable persist dir: use Google Drive on Colab if mounted, else a local folder.
directory_cdb = '/content/drive/MyDrive/chromadb' if os.path.isdir('/content/drive/MyDrive') else './chromadb_medical'
chroma_db = Chroma.from_documents(
    df_document, embed, persist_directory=directory_cdb
)

We are going to create three objects.

* The language model, which can be any of those from OpenAI, the most common being gpt-3.5.
* The memory, responsible for keeping the prompt with all the necessary history.
* The retrieval, used to obtain information stored in ChromaDB.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAI
from langchain_classic.chains.conversation.memory import ConversationBufferWindowMemory
from langchain_classic.chains import RetrievalQA

llm=OpenAI(openai_api_key=OPENAI_API_KEY,
           temperature=0.0)

conversational_memory = ConversationBufferWindowMemory(
    memory_key='chat_history',
    k=4, #Number of messages stored in memory
    return_messages=True #Must return the messages in the response.
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=chroma_db.as_retriever()
)

We can try the isolated Retrieval to see if the information it returns is relevant.




In [ ]:
qa.run("What is the main symptom of LCM?")

Perfect! The information returned is exactly what we desired.

## Creating the Agent.

In [ ]:
from langchain_classic.agents import Tool

#Defining the list of tool objects to be used by LangChain.
tools = [
    Tool(
        name='Medical KB',
        func=qa.run,
        description=(
            """use this tool when answering medical knowledge queries to get
            more information about the topic"""
        )
    )
]

In [ ]:
from langchain_classic.agents import create_react_agent
from langchain_core.prompts import PromptTemplate

# Inline copy of the public "hwchase17/react-chat" prompt. In LangChain 1.x
# hub.pull() of public prompts is disabled (untrusted serialized objects), so
# we define the same ReAct-chat template directly to keep the notebook runnable.
react_chat_template = """Assistant is a large language model trained by OpenAI.

Assistant is designed to be able to assist with a wide range of tasks, from answering simple questions to providing in-depth explanations and discussion on a wide range of topics. As a language model, Assistant is able to generate human-like text based on the input it receives, allowing it to engage in natural-sounding conversations and provide responses that are coherent and relevant to the topic at hand.

Assistant is constantly learning and improving, and its capabilities are constantly evolving. It is able to process and understand large amounts of text, and can use this knowledge to provide accurate and informative responses to a wide range of questions. Additionally, Assistant is able to generate its own text based on the input it receives, allowing it to engage in discussions and provide explanations and descriptions on a wide range of topics.

Overall, Assistant is a powerful system that can help with a wide range of tasks and provide valuable insights and information on a wide range of topics. Whether you need help with a specific question or just want to have a conversation about a particular topic, Assistant is here to assist.

TOOLS:
------

Assistant has access to the following tools:

{tools}

To use a tool, please use the following format:

```
Thought: Do I need to use a tool? Yes
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
```

When you have a response to say to the Human, or if you do not need to use a tool, you MUST use the format:

```
Thought: Do I need to use a tool? No
Final Answer: [your response here]
```

Begin!

Previous conversation history:
{chat_history}

New input: {input}
{agent_scratchpad}"""

prompt = PromptTemplate.from_template(react_chat_template)
agent = create_react_agent(
    tools=tools,
    llm=llm,
    prompt=prompt,
)

In [ ]:
# Create an agent executor by passing in the agent and tools
from langchain_classic.agents import AgentExecutor
agent_executor = AgentExecutor(agent=agent,
                               tools=tools,
                               verbose=True,
                               memory=conversational_memory,
                               max_iterations=30,
                               max_execution_time=600,
                               handle_parsing_errors=True
                               )

### Using the Conversational Agent

To make queries we simply call the `agent` directly.

First i will try a order not related to the Medical field.

In [ ]:
agent_executor.invoke({"input": "Give me the area of square of 2x2"})

Perfect, the model has responded without accessing the configured knowledge database.

Now I will try with a question that is also not related to health.

In [ ]:
agent_executor.invoke({"input": "Do you know who is Clark Kent?"})

It has not accessed either, as the model has been able to identify that it is not a question related to the database that LangChain provides.

Now it's time to try with a question related to Medicine. Let's see if the model can understand that it should first look for information in the vector database at its disposal.

In [ ]:
agent_executor.memory.clear()

In [ ]:
agent_executor.invoke({"input": """I have a patient that can have Botulism,
how can I confirm the diagnosis?"""})

Perfect, the most important thing for us is that it has been able to identify that it should go to the medical database to search for information about the symptoms.

In [ ]:
agent_executor.invoke({"input": "Is this an important illness?"})

And the memory works perfectly. We can maintain a conversation, taking into account that the model knows the previous questions and answers.

# Conclusions.
The experiment has been a small success. The Vectorial database has been configured and filled with information from the dataset. A LangChain agent has been created, and it has been able to retrieve information from the database only when necessary. Don't forget that our ChatBot has memory.

All of this in just a few lines of code!


---